In [ ]:
import torch
import torch.nn as nn

In [ ]:
class SelfAttention(nn.Module):
    """
    Single-Head Scaled Dot-Product Self-Attention.

    Given input X of shape (batch_size, seq_len, d_model), computes:
      Q = X W_Q, K = X W_K, V = X W_V
    then:
      scores = softmax(Q K^T / sqrt(d_k))
      output = scores @ V

    Args:
        d_model: Dimensionality of input embeddings.
        d_k:       Dimensionality of queries/keys (usually d_model).
    """
    def __init__(self, d_model: int, d_k: int = None, dropout: float = 0.1):
        super().__init__()
        d_k = d_model if d_k is None else d_k
        # Learnable projections for Q, K, V:
        self.W_q = nn.Linear(d_model, d_k, bias=False)
        self.W_k = nn.Linear(d_model, d_k, bias=False)
        self.W_v = nn.Linear(d_model, d_k, bias=False)
        self.fc  = nn.Linear(d_k, d_model)
        self.scale = d_k ** 0.5
        self.dropout = nn.Dropout(dropout)
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, X: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:
        """
        Args:
            X:    (batch_size, seq_len, d_model)
            mask: (batch_size, 1, seq_len) or (batch_size, seq_len, seq_len)
                  Mask positions with True will be ignored (set to -inf).
        Returns:
            out:  (batch_size, seq_len, d_model)
        """
        Q = self.W_q(X)  # (B, S, d_k)
        K = self.W_k(X)  # (B, S, d_k)
        V = self.W_v(X)  # (B, S, d_k)

        # scaled dot-product attention scores
        scores = torch.matmul(Q, K.transpose(-2, -1))
        scores = scores / self.scale

        if mask is not None:
            scores = scores.masked_fill(mask, float('-inf'))

        attn = self.softmax(scores)
        attn = self.dropout(attn)

        out = torch.matmul(attn, V)
        out = self.fc(out)
        return out

In [ ]:
if __name__ == "__main__":
    batch_size, seq_len, d_model = 2, 5, 16
    x = torch.randn(batch_size, seq_len, d_model)
    # mask to prevent attending to future tokens
    causal_mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()
    causal_mask = causal_mask.unsqueeze(0)

    sa = SelfAttention(d_model)
    out = sa(x, mask=causal_mask)
    print(f"Input shape : {x.shape}")
    print(f"Output shape: {out.shape}")

Input shape : torch.Size([2, 5, 16])
Output shape: torch.Size([2, 5, 16])
